# 02 — Feature Engineering

## Objectif

Ce notebook prépare les variables explicatives issues du dataset nettoyé
`data_2024_cleaned.csv` en vue de la modélisation des émissions de CO₂ WLTP.

Les objectifs sont :

- analyser la pertinence des variables disponibles après nettoyage ;
- identifier les variables présentant un risque de fuite de données ;
- construire les variables dérivées utiles à la modélisation ;
- analyser la cardinalité des variables catégorielles ;
- définir la stratégie d'encodage à appliquer ultérieurement dans le pipeline ML ;
- produire un dataset de features non encodées, reproductible et exploitable
  pour la séparation train / validation / test.

La variable cible de régression est :

`co2_wltp_g_km`

Les encodeurs statistiques ou catégoriels appris sur les données ne sont pas
ajustés dans ce notebook afin d'éviter toute fuite de données avant la séparation
des jeux d'entraînement et de test.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# Configuration du projet
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "02_feature_engineering":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

INPUT_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "data_2024_cleaned.csv"
)

TARGET = "co2_wltp_g_km"

TEST_MODE = True
NROWS_TEST = 100_000

## 1. Chargement du dataset nettoyé

### Objectif

Cette étape charge le dataset produit par le pipeline de nettoyage.

Deux modes sont conservés :

- **Mode test** : sous-échantillon de 100 000 lignes pour le développement ;
- **Mode complet** : intégralité du dataset nettoyé pour la validation finale.

Aucune transformation n'est appliquée lors du chargement.

In [2]:
if not INPUT_DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Dataset nettoyé introuvable : {INPUT_DATA_PATH}"
    )

if TEST_MODE:
    df = pd.read_csv(
        INPUT_DATA_PATH,
        nrows=NROWS_TEST,
        low_memory=False,
        parse_dates=["registration_date"],
    )

    print(
        f"Mode TEST : {len(df):,} observations chargées."
    )
else:
    df = pd.read_csv(
        INPUT_DATA_PATH,
        low_memory=False,
        parse_dates=["registration_date"],
    )

    print(
        f"Mode COMPLET : {len(df):,} observations chargées."
    )

print(f"Shape : {df.shape}")

Mode TEST : 100,000 observations chargées.
Shape : (100000, 29)


## 2. Contrôle du schéma d'entrée

### Objectif

Avant toute création de features, cette étape vérifie :

- la présence de la variable cible ;
- la présence des principales variables candidates ;
- les types de données ;
- la cohérence du schéma fourni par le notebook 01.

In [3]:
required_columns = {
    "vehicle_record_id",
    "country",
    "manufacturer_name_eu",
    "manufacturer_make",
    "vehicle_category_type",
    "fuel_type",
    "fuel_mode",
    "mass_running_order_kg",
    "engine_capacity_cm3",
    "engine_power_kw",
    "registration_date",
    TARGET,
}

missing_columns = sorted(
    required_columns - set(df.columns)
)

if missing_columns:
    raise ValueError(
        "Variables attendues absentes : "
        + ", ".join(missing_columns)
    )

print("✅ Schéma d'entrée valide.")

✅ Schéma d'entrée valide.


## 3. Séparation entre cible, identifiants et variables explicatives

### Objectif

Toutes les colonnes présentes dans le dataset nettoyé ne doivent pas
nécessairement être utilisées comme variables prédictives.

Cette étape distingue :

- la variable cible ;
- les colonnes de traçabilité ;
- les variables candidates à la modélisation ;
- les variables dont l'utilisation devra être justifiée ou exclue
  pour éviter une fuite de données.

In [4]:
ID_COLUMNS = [
    "vehicle_record_id",
]

TARGET_COLUMNS = [
    TARGET,
]

candidate_features = [
    column
    for column in df.columns
    if column not in ID_COLUMNS + TARGET_COLUMNS
]

print(f"Nombre de variables candidates : {len(candidate_features)}")

Nombre de variables candidates : 27


## 4. Analyse des variables catégorielles

### 4.1 Identification des variables catégorielles

L'objectif est de mesurer la cardinalité des variables catégorielles avant
de choisir une stratégie d'encodage.

Les encodages ne sont pas encore appliqués à ce stade.

In [5]:
categorical_columns = df[
    candidate_features
].select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

categorical_summary = pd.DataFrame(
    {
        "column": categorical_columns,
        "unique_values": [
            df[column].nunique(dropna=False)
            for column in categorical_columns
        ],
        "missing_rate_pct": [
            df[column].isna().mean() * 100
            for column in categorical_columns
        ],
    }
).sort_values(
    "unique_values"
)

display(categorical_summary)

,column,unique_values,missing_rate_pct
12,vehicle_category,1,0.000
11,vehicle_category_type,2,0.000
0,country,3,0.000
14,fuel_mode,6,0.000
13,fuel_type,9,0.000
2,manufacturer_pool,12,4.404
16,emission_standard,32,0.012
15,innovative_technology,73,43.767
4,manufacturer_name_oem,84,0.000
3,manufacturer_name_eu,86,0.000


### 4.2 Classification de la cardinalité

Pour préparer le futur pipeline de preprocessing, les variables
catégorielles sont classées selon leur cardinalité :

- faible cardinalité ;
- cardinalité intermédiaire ;
- forte cardinalité.

Cette classification sert à orienter le choix futur entre One-Hot Encoding,
encodage binaire ou éventuelle suppression / transformation métier.

In [6]:
def classify_cardinality(
    unique_values: int,
) -> str:
    if unique_values <= 15:
        return "low"
    if unique_values <= 100:
        return "medium"
    return "high"


categorical_summary["cardinality_level"] = (
    categorical_summary["unique_values"]
    .apply(classify_cardinality)
)

display(categorical_summary)

,column,unique_values,missing_rate_pct,cardinality_level
12,vehicle_category,1,0.000,low
11,vehicle_category_type,2,0.000,low
0,country,3,0.000,low
14,fuel_mode,6,0.000,low
13,fuel_type,9,0.000,low
2,manufacturer_pool,12,4.404,low
16,emission_standard,32,0.012,medium
15,innovative_technology,73,43.767,medium
4,manufacturer_name_oem,84,0.000,medium
3,manufacturer_name_eu,86,0.000,medium


## 5. Feature Engineering temporel

### 5.1 Extraction des composantes temporelles

La date d'immatriculation n'est pas utilisée directement par les modèles.

Les variables suivantes sont dérivées :

- mois d'immatriculation ;
- composantes cycliques du mois (`sin` / `cos`).

L'année n'est pas encodée cycliquement par défaut : une année n'est pas une
variable périodique. Dans le dataset 2024, elle peut également être constante.

In [7]:
DATE_COLUMN = "registration_date"

df["registration_month"] = (
    df[DATE_COLUMN].dt.month
)

df["registration_month_sin"] = np.sin(
    2
    * np.pi
    * (df["registration_month"] - 1)
    / 12
)

df["registration_month_cos"] = np.cos(
    2
    * np.pi
    * (df["registration_month"] - 1)
    / 12
)

display(
    df[
        [
            DATE_COLUMN,
            "registration_month",
            "registration_month_sin",
            "registration_month_cos",
        ]
    ].head()
)

,registration_date,registration_month,registration_month_sin,registration_month_cos
0,2024-02-21,2,0.5,8.660254e-01
1,2024-12-31,12,-0.5,8.660254e-01
2,2024-10-08,10,-1.0,-1.836970e-16
3,2024-12-30,12,-0.5,8.660254e-01
4,2024-10-16,10,-1.0,-1.836970e-16


### 5.2 Validation des variables temporelles

Cette étape vérifie que les transformations temporelles n'ont introduit
aucune valeur manquante inattendue.

In [8]:
temporal_features = [
    "registration_month",
    "registration_month_sin",
    "registration_month_cos",
]

temporal_missing = (
    df[temporal_features]
    .isna()
    .sum()
)

display(temporal_missing)

if temporal_missing.sum() == 0:
    print("✅ Features temporelles valides.")

registration_month        0
registration_month_sin    0
registration_month_cos    0
dtype: int64

✅ Features temporelles valides.


## 6. Pré-analyse de la stratégie d'encodage

### Objectif

Cette étape propose une première orientation d'encodage à partir de la
cardinalité des variables catégorielles.

Cette orientation est provisoire : certaines variables pourront être exclues
ultérieurement après analyse de leur nature, de leur sémantique ou de leur
pertinence pour la modélisation.

Aucun encodeur n'est ajusté dans cette étape.

In [9]:
encoding_strategy = categorical_summary.copy()

encoding_strategy["recommended_encoding"] = (
    encoding_strategy["cardinality_level"]
    .map(
        {
            "low": "one_hot",
            "medium": "binary_or_frequency",
            "high": "review_or_high_cardinality_encoding",
        }
    )
)

display(
    encoding_strategy[
        [
            "column",
            "unique_values",
            "cardinality_level",
            "recommended_encoding",
        ]
    ]
)

,column,unique_values,cardinality_level,recommended_encoding
12,vehicle_category,1,low,one_hot
11,vehicle_category_type,2,low,one_hot
0,country,3,low,one_hot
14,fuel_mode,6,low,one_hot
13,fuel_type,9,low,one_hot
2,manufacturer_pool,12,low,one_hot
16,emission_standard,32,medium,binary_or_frequency
15,innovative_technology,73,medium,binary_or_frequency
4,manufacturer_name_oem,84,medium,binary_or_frequency
3,manufacturer_name_eu,86,medium,binary_or_frequency


## 7. Analyse de pertinence des variables candidates

### Objectif

Avant de constituer définitivement le dataset destiné à la modélisation,
cette étape examine les variables candidates afin d'identifier :

- les variables à très forte cardinalité ;
- les identifiants ou quasi-identifiants techniques ;
- les variables susceptibles d'introduire une fuite d'information vis-à-vis
  de la cible `co2_wltp_g_km` ;
- les variables nécessitant une justification métier avant leur utilisation.

Cette analyse permet d'identifier les variables nécessitant une analyse
complémentaire avant leur conservation, leur exclusion ou leur traitement
spécifique pour la modélisation.

In [10]:
candidate_analysis = pd.DataFrame({
    "column": [
        column
        for column in df.columns
        if column != TARGET
    ],
    "dtype": [
        str(df[column].dtype)
        for column in df.columns
        if column != TARGET
    ],
    "unique_values": [
        df[column].nunique(dropna=False)
        for column in df.columns
        if column != TARGET
    ],
    "missing_pct": [
        df[column].isna().mean() * 100
        for column in df.columns
        if column != TARGET
    ],
})

candidate_analysis["unique_ratio_pct"] = (
    candidate_analysis["unique_values"]
    / len(df)
    * 100
)

candidate_analysis = candidate_analysis.sort_values(
    "unique_values",
    ascending=False,
)

display(candidate_analysis)

,column,dtype,unique_values,missing_pct,unique_ratio_pct
0,vehicle_record_id,int64,100000,0.000,100.000
9,vehicle_version,object,7068,0.256,7.068
2,vehicle_family_id,object,2990,0.364,2.990
8,vehicle_variant,object,2223,0.232,2.223
11,commercial_name,object,2057,0.003,2.057
15,wltp_test_mass_kg,float64,1958,0.285,1.958
6,type_approval_number,object,1642,0.000,1.642
26,rlfi,object,1546,1.691,1.546
14,mass_running_order_kg,float64,1367,0.000,1.367
27,electric_range_km,float64,543,78.987,0.543


## 8. Analyse de la relation entre certaines variables métier et la cible

### Objectif

Cette étape mesure la relation entre plusieurs variables métier et la cible
`co2_wltp_g_km`.

L'analyse porte en priorité sur :

- `fuel_consumption` ;
- `co2_reduction_wltp_g_km` ;
- `electric_energy_consumption_wh_km` ;
- `electric_range_km`.

Pour chacune de ces variables, les indicateurs suivants sont examinés :

- le nombre d'observations disponibles ;
- le taux de valeurs manquantes ;
- la corrélation linéaire avec la cible.

L'objectif est d'identifier les variables les plus informatives pour la
modélisation et de documenter leur comportement avant la sélection finale
des features.

In [11]:
leakage_candidates = [
    "fuel_consumption",
    "co2_reduction_wltp_g_km",
    "electric_energy_consumption_wh_km",
    "electric_range_km",
]

available_leakage_candidates = [
    column
    for column in leakage_candidates
    if column in df.columns
]

leakage_analysis = pd.DataFrame({
    "column": available_leakage_candidates,
    "non_null_count": [
        df[column].notna().sum()
        for column in available_leakage_candidates
    ],
    "missing_pct": [
        df[column].isna().mean() * 100
        for column in available_leakage_candidates
    ],
    "correlation_with_target": [
        df[[column, TARGET]]
        .corr()
        .iloc[0, 1]
        for column in available_leakage_candidates
    ],
})

leakage_analysis["abs_correlation"] = (
    leakage_analysis["correlation_with_target"].abs()
)

leakage_analysis = leakage_analysis.sort_values(
    "abs_correlation",
    ascending=False,
)

display(leakage_analysis)

,column,non_null_count,missing_pct,correlation_with_target,abs_correlation
0,fuel_consumption,85391,14.609,0.967461,0.967461
3,electric_range_km,21013,78.987,-0.727891,0.727891
2,electric_energy_consumption_wh_km,21038,78.962,0.387561,0.387561
1,co2_reduction_wltp_g_km,56233,43.767,0.156082,0.156082


## 9. Analyse des variables catégorielles candidates à la modélisation

### Objectif

Cette étape examine plus précisément les variables catégorielles encore
présentes dans le dataset avant de définir leur traitement pour la
modélisation.

L'analyse porte notamment sur :

- le nombre de modalités ;
- la nature des valeurs observées ;
- le caractère nominal ou éventuellement ordinal des modalités ;
- la présence de codes, références ou identifiants techniques ;
- la redondance éventuelle entre plusieurs variables décrivant une information
  similaire.

Aucune variable n'est exclue à ce stade. Les résultats de cette analyse
serviront à établir la sélection finale des variables dans l'étape suivante.

In [12]:
categorical_analysis = []

for column in categorical_columns:
    non_null_values = (
        df[column]
        .dropna()
        .astype(str)
    )

    categorical_analysis.append({
        "variable": column,
        "n_unique": non_null_values.nunique(),
        "missing_pct": round(
            df[column].isna().mean() * 100,
            2,
        ),
        "sample_values": non_null_values.unique()[:10].tolist(),
    })

categorical_analysis_df = (
    pd.DataFrame(categorical_analysis)
    .sort_values(
        "n_unique",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(categorical_analysis_df)

,variable,n_unique,missing_pct,sample_values
0,vehicle_version,7067,0.26,"[5CFETNA5PAX, FD7FD7CW0094BIA1BI0, R25W3620000..."
1,vehicle_family_id,2989,0.36,"[IP-BX72_2021_00005-WF0-1, IP-MQB27SZ_A1_1021-..."
2,vehicle_variant,2222,0.23,"[B7JG12X, ACDLAA, DH2, AH2S, RKJ, MXPJ10(H), B..."
3,commercial_name,2056,0.00,"[PUMA, TAIGO, RAFALE, S-CROSS, KANGOO E-TECH E..."
4,type_approval_number,1642,0.00,"[e9*2007/46*3165*15, e13*2018/858*00140*06, e9..."
5,rlfi,1545,1.69,"[RL-B479_2019_00001-WF0-1, RL-DQ200_7F_17_012-..."
6,vehicle_type,509,0.02,"[J2K, CS, RHN, JY, RFK, XPB1F(M), DJF, RJB, AW..."
7,manufacturer_make,98,0.00,"[FORD, VOLKSWAGEN, RENAULT, SUZUKI, TOYOTA, DA..."
8,manufacturer_name_eu,86,0.00,"[FORD WERKE GMBH, VOLKSWAGEN, RENAULT, MAGYAR ..."
9,manufacturer_name_oem,84,0.00,"[FORD-WERKE GMBH, VOLKSWAGEN AG, RENAULT SAS, ..."


### 9.1 Analyse complémentaire des variables à forte cardinalité

#### Objectif

La section 9 a déjà permis d'identifier les variables catégorielles présentant
une forte cardinalité et d'observer des exemples de leurs modalités.

Cette étape complète cette analyse en évaluant uniquement la structure des
valeurs des variables à forte cardinalité afin de déterminer si elles
correspondent principalement à :

- des références techniques ou administratives ;
- des codes alphanumériques ;
- ou des catégories métier directement interprétables.

Cette analyse complète les observations de la section 9 sans recalculer les
indicateurs déjà étudiés.

In [13]:
import re


high_cardinality_candidates = [
    "vehicle_family_id",
    "type_approval_number",
    "vehicle_type",
    "vehicle_variant",
    "vehicle_version",
    "commercial_name",
    "rlfi",
]


def is_code_like(value: str) -> bool:
    value = str(value).strip()

    if not value:
        return False

    has_digit = bool(re.search(r"\d", value))
    has_separator = bool(re.search(r"[-_*./]", value))
    has_letter = bool(re.search(r"[A-Za-z]", value))

    return has_digit and (
        has_separator
        or has_letter
    )


structure_analysis = []

for column in high_cardinality_candidates:
    values = (
        df[column]
        .dropna()
        .astype(str)
        .str.strip()
        .drop_duplicates()
    )

    code_like_pct = (
        values.apply(is_code_like).mean() * 100
        if len(values) > 0
        else 0
    )

    structure_analysis.append({
        "variable": column,
        "code_like_pct": round(code_like_pct, 2),
        "avg_value_length": round(
            values.str.len().mean(),
            2,
        ),
    })

structure_analysis_df = (
    pd.DataFrame(structure_analysis)
    .sort_values(
        "code_like_pct",
        ascending=False,
    )
    .reset_index(drop=True)
)

display(structure_analysis_df)

,variable,code_like_pct,avg_value_length
0,type_approval_number,100.00,18.94
1,vehicle_family_id,99.87,19.75
2,rlfi,99.74,18.88
3,vehicle_version,98.61,18.97
4,vehicle_variant,91.85,6.09
5,commercial_name,78.79,16.29
6,vehicle_type,62.28,4.24


### 9.2 Analyse de la redondance des variables relatives au constructeur

#### Objectif

Le dataset contient plusieurs variables décrivant le constructeur du véhicule :

- `manufacturer_pool` ;
- `manufacturer_name_eu` ;
- `manufacturer_name_oem` ;
- `manufacturer_make`.

La section 9 a montré que ces variables présentent des cardinalités différentes,
mais cette information ne permet pas à elle seule de déterminer si elles sont
complémentaires ou redondantes.

Cette étape analyse leurs relations afin d'évaluer si certaines variables
portent essentiellement la même information sous des libellés différents.

L'analyse repose sur le nombre moyen de modalités d'une variable associées à
chaque modalité d'une autre variable constructeur.

Une relation proche de **1 pour 1** indique qu'une variable détermine presque
toujours l'autre et constitue donc un indice de redondance.

Aucune variable n'est supprimée dans cette étape. Les résultats seront utilisés
pour établir la sélection finale des variables.

In [14]:
manufacturer_columns = [
    "manufacturer_pool",
    "manufacturer_name_eu",
    "manufacturer_name_oem",
    "manufacturer_make",
]


def analyze_categorical_relationship(
    df: pd.DataFrame,
    source: str,
    target: str,
) -> dict:
    """
    Analyse combien de modalités distinctes de `target`
    sont associées à chaque modalité de `source`.
    """

    relationships = (
        df[[source, target]]
        .dropna()
        .drop_duplicates()
        .groupby(source)[target]
        .nunique()
    )

    return {
        "source": source,
        "target": target,
        "mean_targets_per_source": round(
            relationships.mean(),
            3,
        ),
        "max_targets_per_source": int(
            relationships.max()
        ),
        "pct_source_with_one_target": round(
            (relationships == 1).mean() * 100,
            2,
        ),
    }


manufacturer_relationships = []

for source in manufacturer_columns:
    for target in manufacturer_columns:

        if source == target:
            continue

        manufacturer_relationships.append(
            analyze_categorical_relationship(
                df=df,
                source=source,
                target=target,
            )
        )


manufacturer_relationships_df = pd.DataFrame(
    manufacturer_relationships
)

display(manufacturer_relationships_df)

,source,target,mean_targets_per_source,max_targets_per_source,pct_source_with_one_target
0,manufacturer_pool,manufacturer_name_eu,4.455,9,0.00
1,manufacturer_pool,manufacturer_name_oem,4.455,9,0.00
2,manufacturer_pool,manufacturer_make,4.273,9,18.18
3,manufacturer_name_eu,manufacturer_pool,1.000,1,100.00
4,manufacturer_name_eu,manufacturer_name_oem,1.128,12,98.84
5,manufacturer_name_eu,manufacturer_make,1.860,32,73.26
6,manufacturer_name_oem,manufacturer_pool,1.000,1,100.00
7,manufacturer_name_oem,manufacturer_name_eu,1.155,3,85.71
8,manufacturer_name_oem,manufacturer_make,1.786,32,73.81
9,manufacturer_make,manufacturer_pool,1.000,1,100.00


### 9.3 Analyse de la relation entre les variables constructeur et la cible

#### Objectif

La section 9.2 a permis d'étudier les relations entre les différentes variables
décrivant le constructeur. Cette analyse de redondance ne permet toutefois pas,
à elle seule, d'évaluer leur relation avec la cible `co2_wltp_g_km`.

Cette section étudie donc la relation entre les variables constructeur encore
candidates :

- `manufacturer_pool` ;
- `manufacturer_make` ;

et la variable cible `co2_wltp_g_km`.

L'analyse est réalisée en deux étapes :

1. mesure de l'association globale avec la cible à l'aide du rapport de
   corrélation η² ;
2. analyse descriptive des émissions de CO₂ selon les différentes modalités
   de chaque variable constructeur.

Aucune variable n'est supprimée dans cette section. Les résultats obtenus
seront utilisés lors de la sélection finale des variables.

#### 9.3.1 Mesure de l'association avec la cible par le rapport de corrélation η²

Le rapport de corrélation η² (eta carré) est utilisé pour mesurer l'association
entre une variable catégorielle et la cible numérique `co2_wltp_g_km`.

Pour chaque variable constructeur étudiée, η² mesure la part de la variabilité
de la cible associée aux groupes définis par ses différentes modalités.

La valeur de η² est comprise entre 0 et 1 :

- une valeur proche de 0 indique une faible association entre les groupes et
  la cible ;
- une valeur plus élevée indique une association plus importante.

Cette mesure ne constitue pas une preuve de causalité et ne détermine pas, à
elle seule, si une variable doit être conservée ou supprimée.

In [15]:
def correlation_ratio(
    categories: pd.Series,
    values: pd.Series,
) -> float:
    """
    Calcule le rapport de corrélation eta carré entre
    une variable catégorielle et une variable numérique.
    """

    valid_mask = (
        categories.notna()
        & values.notna()
    )

    categories_valid = categories[valid_mask]
    values_valid = values[valid_mask]

    overall_mean = values_valid.mean()

    total_variation = (
        (values_valid - overall_mean) ** 2
    ).sum()

    if total_variation == 0:
        return 0.0

    between_group_variation = 0.0

    for _, group_values in values_valid.groupby(
        categories_valid
    ):
        between_group_variation += (
            len(group_values)
            * (
                group_values.mean()
                - overall_mean
            ) ** 2
        )

    return (
        between_group_variation
        / total_variation
    )


manufacturer_target_analysis = []

for column in [
    "manufacturer_pool",
    "manufacturer_make",
]:
    eta_squared = correlation_ratio(
        df[column],
        df[TARGET],
    )

    manufacturer_target_analysis.append(
        {
            "variable": column,
            "n_unique": df[column].nunique(),
            "eta_squared": round(
                eta_squared,
                4,
            ),
        }
    )


manufacturer_target_analysis_df = pd.DataFrame(
    manufacturer_target_analysis
)

display(manufacturer_target_analysis_df)

,variable,n_unique,eta_squared
0,manufacturer_pool,11,0.0228
1,manufacturer_make,98,0.1561


#### 9.3.2 Analyse descriptive des émissions de CO₂ par constructeur

Cette analyse complète le rapport de corrélation η² en examinant directement
la distribution des émissions de CO₂ selon les modalités de
`manufacturer_pool` et `manufacturer_make`.

Pour chaque modalité, trois indicateurs sont présentés :

- `observations` : nombre de véhicules observés ;
- `mean_co2` : émission moyenne de CO₂ ;
- `median_co2` : émission médiane de CO₂.

Ces statistiques permettent d'observer les différences de niveaux d'émission
entre les groupes et de tenir compte de leur représentativité dans le dataset.

Pour `manufacturer_make`, seules les modalités les plus représentées sont
affichées afin de conserver une lecture synthétique du tableau.

Cette analyse descriptive complète η² sans introduire de nouvelle décision de
sélection. La décision de conservation ou d'exclusion des variables sera
formalisée dans la section suivante.

In [16]:
for column in [
    "manufacturer_pool",
    "manufacturer_make",
]:
    manufacturer_stats = (
        df[
            [
                column,
                TARGET,
            ]
        ]
        .dropna(
            subset=[column, TARGET]
        )
        .groupby(column)[TARGET]
        .agg(
            observations="count",
            mean_co2="mean",
            median_co2="median",
        )
        .sort_values(
            "observations",
            ascending=False,
        )
    )

    print(f"\n{column}")

    display(
        manufacturer_stats.head(20)
    )


manufacturer_pool


,observations,mean_co2,median_co2
manufacturer_pool,,,
VOLKSWAGEN,30737,122.261964,132.0
STELLANTIS,17197,104.887190,123.0
RENAULT-NISSAN-MITSUBISHI,14898,109.468318,118.0
BMW,7446,108.328633,135.0
SUBARU-MAZDA-TOYOTA,6644,110.902769,108.0
MERCEDES-BENZ AG,6542,108.772394,137.0
HYUNDAI MOTOR EUROPE,3470,112.850432,124.0
VOLVO CARS POLESTAR SUZUKI,2911,85.478186,106.0
FORD,2864,124.361034,128.0



manufacturer_make


,observations,mean_co2,median_co2
manufacturer_make,,,
VOLKSWAGEN VW,10665,124.744210,134.0
RENAULT,7974,101.529345,109.0
PEUGEOT,6758,101.503995,120.0
BMW,6412,109.805053,134.0
SKODA,5933,116.564638,126.0
AUDI,5490,131.956102,143.0
MERCEDES-BENZ,5059,120.957699,140.0
TOYOTA,4994,105.991590,106.0
DACIA,4821,118.527484,124.0


### 9.4 Inspection sémantique des modalités des variables catégorielles candidates

#### Objectif

Les analyses quantitatives précédentes permettent d'étudier notamment la
cardinalité, la fréquence des modalités, la redondance entre certaines
variables et leur relation avec la cible.

Ces indicateurs ne permettent cependant pas, à eux seuls, d'évaluer la
signification métier des valeurs contenues dans une variable catégorielle.

Cette étape complète donc l'analyse par une inspection des modalités de
l'ensemble des variables catégorielles encore candidates à la modélisation.

L'objectif est notamment d'identifier :

- les catégories dont les modalités possèdent une signification métier
  directement interprétable ;
- les codes techniques ou administratifs dont la signification n'est pas
  exploitable à partir des informations disponibles dans l'étude ;
- les variables textuelles ou semi-structurées combinant plusieurs informations
  dans une même modalité ;
- les éventuelles variables nécessitant un référentiel externe avant de pouvoir
  être exploitées correctement.

Cette inspection ne réalise aucune suppression automatique. Elle fournit les
éléments nécessaires à la décision finale de sélection des variables.

In [17]:
categorical_columns = df.select_dtypes(
    include=["object", "category"]
).columns.tolist()

MAX_MODALITIES_DISPLAYED = 30

for column in categorical_columns:
    n_unique = df[column].nunique(dropna=True)
    missing_rate = df[column].isna().mean() * 100

    modalities = (
        df[column]
        .dropna()
        .astype(str)
        .unique()
    )

    print("=" * 100)
    print(f"Variable          : {column}")
    print(f"Modalités uniques : {n_unique:,}")
    print(f"Valeurs manquantes: {missing_rate:.2f} %")
    print()

    print(
        f"Exemples de modalités "
        f"(maximum {MAX_MODALITIES_DISPLAYED}) :"
    )

    print(
        modalities[
            :MAX_MODALITIES_DISPLAYED
        ].tolist()
    )

    if n_unique > MAX_MODALITIES_DISPLAYED:
        print("...")

    print() 

Variable          : country
Modalités uniques : 3
Valeurs manquantes: 0.00 %

Exemples de modalités (maximum 30) :
['FR', 'DE', 'SI']

Variable          : vehicle_family_id
Modalités uniques : 2,989
Valeurs manquantes: 0.36 %

Exemples de modalités (maximum 30) :
['IP-BX72_2021_00005-WF0-1', 'IP-MQB27SZ_A1_1021-WVW-1', 'IP-HNA1M2PDB1A_000-VF1', 'IP-6_004593-TSM-1', 'IP-FKA1JAE044A_000-VF1', 'IP-0153-JT1', 'IP-JFB1M6PJT4A_000-UU1-0', 'IP-HNC1M4P021A_000-VF1', 'IP-JBA1MUP001A_002-VF1-1', 'IP-JFE1MTPJT4A_000-UU1', 'IP-MQB27SZ_A1_1020-WVW-1', 'IP-MQB27ZZ_B2_0534-WVW-1', 'IP-JFA1M6PJH3B_000-UU1-0', 'IP-HNS____ATN85447-VR3', 'IP-JFD1MDPJT4B_000-UU1', 'IP-2020_8419-W1K-1', 'IP-JFB1MTGJT4A_000-UU1-0', 'IP-4_1228-JSA-1', 'IP-JAA1MTPJT4A_000-VF1', 'IP-JAB1N8H0010_000-VF1', 'IP-MQB48ZZ_B0_1279-WVW', 'IP-MLB42AZ_A0_0732-WAU-1', 'IP-JFA1MGPJH3A_000-UU1', 'IP-0112-JT1-1', 'IP-HNS____AT6_1436-W0V-0', 'IP-0500769-TMA-1', 'IP-MQB27ZZ_A0_1019-TMB-1', 'IP-HNK____MB6_5422-VR3', 'IP-6_00458-TSM-1', 'IP-13_

### 9.5 Conclusion de l'inspection sémantique des variables catégorielles

#### Objectif

L'inspection réalisée à la section 9.4 complète les analyses quantitatives
précédentes par l'examen direct de la nature et de la signification des
modalités des variables catégorielles.

Cette analyse permet de distinguer :

- les variables décrivant des caractéristiques directement exploitables du
  véhicule ;
- les variables de contexte qui ne décrivent pas une caractéristique
  intrinsèque du véhicule ;
- les références techniques ou administratives ;
- les codes non suffisamment interprétables ;
- les variables textuelles ou semi-structurées dont l'exploitation brute
  n'est pas adaptée au modèle principal.

#### Variables catégorielles conservées

Les variables suivantes présentent une information métier directement
exploitable pour caractériser le véhicule :

- `manufacturer_make` : marque du véhicule ;
- `vehicle_category_type` : catégorie réglementaire du véhicule (`M1`, `N1`) ;
- `fuel_type` : type de carburant ou d'énergie ;
- `fuel_mode` : mode énergétique du véhicule.

Ces quatre variables sont conservées comme variables explicatives candidates.

#### Variable de contexte géographique exclue

- `country` représente le pays associé à l'observation.

Le pays n'est pas une caractéristique physique, énergétique ou technique
intrinsèque du véhicule déterminant directement ses émissions de CO₂.

Une éventuelle relation statistique entre `country` et `co2_wltp_g_km` peut
notamment refléter des différences de composition des parcs automobiles entre
pays plutôt qu'un mécanisme propre au véhicule.

Afin que le modèle repose sur les caractéristiques du véhicule et puisse être
appliqué indépendamment du pays représenté dans les données d'apprentissage,
`country` n'est pas retenue comme variable explicative.

#### Variables catégorielles non exploitables retenues pour exclusion

Les variables suivantes ne sont pas retenues :

- `vehicle_family_id` : référence technique de famille de véhicule à forte
  cardinalité ;
- `type_approval_number` : référence administrative d'homologation ;
- `vehicle_type` : codes techniques hétérogènes sans signification directement
  exploitable dans l'étude ;
- `vehicle_variant` : codes techniques de variante à forte cardinalité ;
- `vehicle_version` : codes techniques de version à très forte cardinalité ;
- `rlfi` : références techniques à forte cardinalité ;
- `commercial_name` : information textuelle semi-structurée à forte
  cardinalité, combinant de manière hétérogène modèle, version, motorisation
  et autres informations commerciales ;
- `innovative_technology` : codes et combinaisons de codes techniques non
  interprétables sans référentiel, avec une proportion importante de valeurs
  manquantes ;
- `emission_standard` : modalités hétérogènes mêlant des libellés
  interprétables et des codifications qui ne peuvent pas être exploitées de
  manière homogène et fiable dans l'étude.

Au total, dix variables catégorielles sont donc exclues lors de la sélection
finale réalisée à la section 10.

Les quatre variables catégorielles conservées sont :

- `manufacturer_make` ;
- `vehicle_category_type` ;
- `fuel_type` ;
- `fuel_mode`.

## 10. Sélection finale des variables pour la modélisation

### Objectif

Cette étape centralise et applique les décisions de sélection établies au cours
des analyses précédentes.

Aucune nouvelle analyse n'est réalisée dans cette section.

Les exclusions sont regroupées selon leur justification :

- variable de contexte géographique ;
- variables catégorielles non retenues après inspection sémantique ;
- variables constructeur redondantes ou peu informatives ;
- identifiant technique ;
- variables temporelles non conservées ;
- variable constante.

### 10.1 Variable de contexte géographique

Conformément à la conclusion de la section 9.5 :

- `country` est exclue.

Cette variable identifie le pays associé à l'observation mais ne constitue pas
une caractéristique intrinsèque du véhicule déterminant directement ses
émissions de CO₂.

Son exclusion permet également d'éviter que le modèle apprenne des différences
de composition des parcs automobiles propres aux pays présents dans les données
plutôt que les relations entre les caractéristiques du véhicule et les
émissions.

### 10.2 Variables catégorielles exclues après inspection sémantique

Conformément à la conclusion de la section 9.5, les variables suivantes sont
exclues :

- `vehicle_family_id` ;
- `type_approval_number` ;
- `vehicle_type` ;
- `vehicle_variant` ;
- `vehicle_version` ;
- `rlfi` ;
- `commercial_name` ;
- `innovative_technology` ;
- `emission_standard`.

Les variables catégorielles conservées sont :

- `manufacturer_make` ;
- `vehicle_category_type` ;
- `fuel_type` ;
- `fuel_mode`.

### 10.3 Variables constructeur non retenues

Les analyses des sections 9.2 et 9.3 ont conduit à ne pas retenir :

- `manufacturer_name_eu` ;
- `manufacturer_name_oem` ;
- `manufacturer_pool`.

`manufacturer_name_eu` et `manufacturer_name_oem` apportent des
représentations détaillées et fortement liées de l'information constructeur.

`manufacturer_pool` représente un niveau plus agrégé du constructeur et
présente une faible association avec la cible dans les données analysées.

`manufacturer_make` est conservée comme représentation exploitable de la
marque du véhicule.

### 10.4 Identifiant technique

- `vehicle_record_id` est exclue car elle constitue un identifiant technique de
  traçabilité et non une caractéristique explicative du véhicule.

### 10.5 Variables temporelles non conservées

- `registration_date` est exclue après création de
  `registration_month_sin` et `registration_month_cos` ;
- `registration_month` est une variable intermédiaire utilisée pour construire
  ces deux composantes cycliques et n'est pas conservée.

### 10.6 Variable constante

- `vehicle_category` est exclue car elle ne possède qu'une seule modalité dans
  les données analysées et n'apporte donc aucune variation exploitable.

### 10.7 Décision finale

L'ensemble des variables explicitement exclues dans cette section est retiré du
dataset de features.

Les autres variables sont conservées pour la suite du pipeline.

Les opérations d'imputation, d'encodage et les transformations propres à la
préparation des données pour l'apprentissage seront réalisées dans l'étape de
preprocessing Machine Learning.

In [ ]:
# ---------------------------------------------------------------------
# Sélection finale des variables
# ---------------------------------------------------------------------

COLUMNS_TO_EXCLUDE = {
    # Variable de contexte géographique
    "country",

    # Identifiant technique
    "vehicle_record_id",

    # Références techniques ou administratives
    "vehicle_family_id",
    "type_approval_number",
    "vehicle_type",
    "vehicle_variant",
    "vehicle_version",
    "rlfi",

    # Variable textuelle composite à forte cardinalité
    "commercial_name",

    # Variables non interprétables / non homogènes
    "innovative_technology",
    "emission_standard",

    # Variables constructeur non retenues
    "manufacturer_pool",
    "manufacturer_name_eu",
    "manufacturer_name_oem",

    # Variables temporelles non conservées
    "registration_date",
    "registration_month",

    # Variable constante
    "vehicle_category",
}

feature_columns = [
    column
    for column in df.columns
    if column not in COLUMNS_TO_EXCLUDE
]

df_features = df[
    feature_columns
].copy()

print(
    f"Dataset de features : "
    f"{len(df_features):,} observations × "
    f"{df_features.shape[1]} variables"
)

print("\nVariables exclues :")

for column in sorted(COLUMNS_TO_EXCLUDE):
    print(f"  - {column}") 

Dataset de features : 100,000 observations × 15 variables

Variables exclues :
  - commercial_name
  - country
  - emission_standard
  - innovative_technology
  - manufacturer_name_eu
  - manufacturer_name_oem
  - manufacturer_pool
  - registration_date
  - registration_month
  - rlfi
  - type_approval_number
  - vehicle_category
  - vehicle_family_id
  - vehicle_record_id
  - vehicle_type
  - vehicle_variant
  - vehicle_version


## 11. Contrôles qualité du dataset de features

### Objectif

Cette étape vérifie que le dataset obtenu respecte les décisions de sélection
définies et appliquées à la section **10. Sélection finale des variables pour
la modélisation**.

Les contrôles portent sur :

- la présence de la variable cible `co2_wltp_g_km` ;
- l'unicité des noms de colonnes ;
- l'absence de toutes les variables exclues à la section 10 ;
- la présence des features temporelles `registration_month_sin` et
  `registration_month_cos` ;
- l'absence de valeurs manquantes dans ces deux features temporelles.

Ces contrôles ne modifient pas le dataset et n'introduisent aucune nouvelle
règle de sélection.

In [19]:
quality_checks = {
    "Variable cible présente":
        TARGET in df_features.columns,

    "Noms de colonnes uniques":
        df_features.columns.is_unique,

    "Toutes les variables exclues sont absentes":
        COLUMNS_TO_EXCLUDE.isdisjoint(
            df_features.columns
        ),

    "Feature mois sinus présente":
        "registration_month_sin"
        in df_features.columns,

    "Feature mois cosinus présente":
        "registration_month_cos"
        in df_features.columns,

    "Features temporelles sans valeur manquante":
        not df_features[
            [
                "registration_month_sin",
                "registration_month_cos",
            ]
        ].isna().any().any(),
}

failed_checks = []

for check, result in quality_checks.items():
    status = "✅" if result else "❌"
    print(f"{status} {check}")

    if not result:
        failed_checks.append(check)

if failed_checks:
    raise ValueError(
        "Contrôles qualité échoués : "
        + ", ".join(failed_checks)
    )

print("✅ Dataset de features validé.")

✅ Variable cible présente
✅ Noms de colonnes uniques
✅ Toutes les variables exclues sont absentes
✅ Feature mois sinus présente
✅ Feature mois cosinus présente
✅ Features temporelles sans valeur manquante
✅ Dataset de features validé.


## 12. Export du dataset de Feature Engineering

Le dataset produit à cette étape n'est pas encore encodé par des
transformations apprises sur les données.

Il constitue l'entrée du futur pipeline de séparation train / test et de
préprocessing Machine Learning.

In [20]:
OUTPUT_PATH = (
    PROJECT_ROOT
    / "data"
    / "interim"
    / (
        "data_2024_features_test.csv"
        if TEST_MODE
        else "data_2024_features.csv"
    )
)

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True,
)

df_features.to_csv(
    OUTPUT_PATH,
    index=False,
)

print("✅ Dataset de features exporté.")
print(f"Fichier      : {OUTPUT_PATH}")
print(f"Observations : {len(df_features):,}")
print(f"Variables    : {df_features.shape[1]}")

✅ Dataset de features exporté.
Fichier      : /home/jmbandong/projects/ml-projects/vehicle-emissions-prediction-mlops/data/interim/data_2024_features_test.csv
Observations : 100,000
Variables    : 15


## 13. Vérification finale du dataset de features

### Objectif

Cette étape réalise un contrôle final du dataset de features généré avant de
poursuivre vers la séparation des jeux d'entraînement et de test.

Les vérifications portent sur :

- l'aperçu des premières observations ;
- la structure générale du DataFrame ;
- le nombre final d'observations et de variables ;
- la présence de la variable cible ;
- la cohérence du schéma obtenu après Feature Engineering.

Cette étape ne modifie pas les données. Elle constitue uniquement un contrôle
de validation du dataset produit.

In [21]:
display(df_features.head())

print("\nInformations générales du dataset de features :")
df_features.info()

print("\nRésumé final :")
print(f"Shape finale          : {df_features.shape}")
print(
    f"Variable cible présente : "
    f"{TARGET in df_features.columns}"
)

,manufacturer_make,vehicle_category_type,mass_running_order_kg,wltp_test_mass_kg,co2_wltp_g_km,fuel_type,fuel_mode,engine_capacity_cm3,engine_power_kw,electric_energy_consumption_wh_km,co2_reduction_wltp_g_km,fuel_consumption,electric_range_km,registration_month_sin,registration_month_cos
0,FORD,M1,1315.0,1405.0,121.0,e85,H,999.0,91.0,NaN,2.00,5.4,NaN,0.5,8.660254e-01
1,VOLKSWAGEN,M1,1255.0,1403.0,137.0,petrol,M,999.0,81.0,NaN,1.17,6.0,NaN,-0.5,8.660254e-01
2,RENAULT,M1,1735.0,1875.0,113.0,petrol,H,1199.0,96.0,NaN,0.68,5.0,NaN,-1.0,-1.836970e-16
3,SUZUKI,M1,1365.0,1443.0,116.0,petrol,H,1462.0,75.0,NaN,0.81,5.1,NaN,-0.5,8.660254e-01
4,RENAULT,M1,1871.0,1978.0,0.0,electric,E,NaN,90.0,194.0,NaN,NaN,280.0,-1.0,-1.836970e-16



Informations générales du dataset de features :
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 15 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   manufacturer_make                  99996 non-null   object 
 1   vehicle_category_type              100000 non-null  object 
 2   mass_running_order_kg              100000 non-null  float64
 3   wltp_test_mass_kg                  99715 non-null   float64
 4   co2_wltp_g_km                      100000 non-null  float64
 5   fuel_type                          100000 non-null  object 
 6   fuel_mode                          100000 non-null  object 
 7   engine_capacity_cm3                85805 non-null   float64
 8   engine_power_kw                    99999 non-null   float64
 9   electric_energy_consumption_wh_km  21038 non-null   float64
 10  co2_reduction_wltp_g_km            56233 non-null   floa